# Бинарная модель CatBoost
## 1. Подготовка данных
Читаем готовую кадровую витрину `data/teachers.parquet`: одна строка — педагог в году.
Цель `target=1` — отсутствие в следующем срезе системы, `target=0` — присутствие,
включая переход в другую школу. В 2024 исход неизвестен.

Пока выполняем только подготовку: обучение, Optuna и сохранение модели будут следующими этапами.
Запускать из корня проекта или папки `prototype/`, выбрав Python из `.venv`.

In [1]:
from pathlib import Path
import pandas as pd
from sklearn.model_selection import train_test_split

root = Path.cwd()
if not (root / 'data').is_dir():
    root = root.parent

df = pd.read_parquet(root / 'data/teachers.parquet')
assert not df.duplicated(['id', 'year']).any()
display(df.groupby('year')['target'].agg(['size', 'count', 'mean']))

,size,count,mean
year,,,
2022,17396,17396,0.149230
2023,17394,17394,0.132172
2024,17933,0,NaN


### Признаки
Все нужные признаки уже есть в parquet. Выбираем их здесь, без импорта кода дашборда.
Идентификаторы и исходы в признаки не входят. Числовые признаки включают бинарные 0/1.

Категориальные столбцы в готовой витрине уже строковые и не имеют пропусков.
Числовые NaN сохраняем: CatBoost умеет с ними работать. Дополнительные признаки,
One-Hot Encoding, масштабирование и заполнение пропусков здесь не нужны.

In [2]:
numeric = [
    'age', 'experience_age_pedagogic_clean', 'experience_age_other_clean',
    'n_subjects', 'n_parallels', 'min_parallel', 'max_parallel',
    'sex', 'class_manager', 'edu_higher', 'edu_pedagogical', 'edu_speciality_match',
    'pk_recent', 'pk_this_year', 'pk_hours_3y', 'retraining',
]
categorical = ['main_position_cat', 'academic_degree_cat', 'category_qual_cat', 'district', 'speciality']
features = numeric + categorical

assert df[categorical].notna().all().all()
display(df[features].dtypes.to_frame('Тип признака'))

,Тип признака
age,int64
experience_age_pedagogic_clean,int64
experience_age_other_clean,int64
n_subjects,int64
n_parallels,int64
min_parallel,int64
max_parallel,int64
sex,int64
class_manager,int64
edu_higher,int64


### Разделение по годам
2022 — обучение, 2023 — итоговый тест. Данные 2024 готовим отдельно для будущего прогноза,
не заполняя неизвестную цель нулями. Ключи `id, year` сохраняем для сопоставления прогнозов.

In [3]:
train = df[df['year'] == 2022]
test = df[df['year'] == 2023]
forecast = df[df['year'] == 2024]

assert not train.empty and not test.empty and not forecast.empty
assert train['target'].isin([0, 1]).all() and test['target'].isin([0, 1]).all()
assert forecast['target'].isna().all()

X_train, y_train = train[features].copy(), train['target'].astype(int)
X_test, y_test = test[features].copy(), test['target'].astype(int)
X_forecast = forecast[features].copy()
forecast_keys = forecast[['id', 'year']].copy()

display(X_train.isna().sum().loc[lambda s: s > 0].to_frame('Пропусков в train'))

,Пропусков в train
edu_speciality_match,16904


### Validation для будущего подбора параметров
Как в HW8, выделяем 20% 2022 года с сохранением доли классов.
`X_fit, y_fit` будут использоваться для обучения при подборе,
`X_valid, y_valid` — для Optuna и early stopping.
2023 не участвует в выборе параметров; 2024 не участвует ни в обучении, ни в проверке.

In [4]:
X_fit, X_valid, y_fit, y_valid = train_test_split(
    X_train, y_train, test_size=0.2, stratify=y_train, random_state=42,
)

display(pd.DataFrame({
    'Строк': [len(X_fit), len(X_valid), len(X_test), len(X_forecast)],
    'Признаков': [len(features)] * 4,
    'Доля уходов': [y_fit.mean(), y_valid.mean(), y_test.mean(), pd.NA],
}, index=['Обучение 2022', 'Validation 2022', 'Тест 2023', 'Прогноз 2024']))

,Строк,Признаков,Доля уходов
Обучение 2022,13916,21,0.149253
Validation 2022,3480,21,0.149138
Тест 2023,17394,21,0.132172
Прогноз 2024,17933,21,<NA>


Данные готовы. Для следующего этапа доступны `X_fit`, `X_valid`, `X_train`, `X_test`,
соответствующие цели `y_fit`, `y_valid`, `y_train`, `y_test`, а также `X_forecast` и `forecast_keys`.
Список признаков — `features`, категориальных столбцов для CatBoost — `categorical`.
Обучение, Optuna и сохранение будут реализованы ниже в этом же ноутбуке.
На текущем этапе никакие данные или файлы модели не записываются.

# Модель: прогноз ухода педагога (target)

**Задача:** бинарная классификация — уйдёт педагог в следующем году или останется.  
**Модель:** CatBoost Classifier.  
**Сплит:** временной (обучаемся на ранних годах, валидируем на поздних).  
**Данные:** `teachers.parquet`, зерно `id + year`.

In [5]:
import pandas as pd
import numpy as np
from catboost import CatBoostClassifier, Pool
from sklearn.metrics import (
    classification_report,
    roc_auc_score,
    precision_recall_curve,
    auc as pr_auc,
)
from pathlib import Path
import joblib
import warnings

warnings.filterwarnings("ignore")

In [6]:
DATA_PATH = Path("data/teachers.parquet")
MODEL_DIR = Path("models")
MODEL_DIR.mkdir(exist_ok=True)

root = Path.cwd()
if not (root / 'data').is_dir():
    root = root.parent

df = pd.read_parquet(root / DATA_PATH)
print(f"Форма: {df.shape}")
print(f"Годы: {sorted(df['year'].unique())}")
print(f"Пропуски target: {df['target'].isna().sum()}")

Форма: (52723, 35)
Годы: [np.int64(2022), np.int64(2023), np.int64(2024)]
Пропуски target: 17933


## Целевая переменная

`target` — бинарный признак: 1 = педагог ушёл, 0 = остался.  
Заполнен только для тех лет, где есть наблюдение следующего года.  
Последний год в данных всегда без target (некуда смотреть вперёд).

In [7]:
df_model = df[df["target"].notna()].copy()
df_model["target"] = df_model["target"].astype(int)

print(f"Строк для обучения: {len(df_model)}")
print(df_model["target"].value_counts(normalize=True))

Строк для обучения: 34790
target
0    0.859299
1    0.140701
Name: proportion, dtype: float64


## Отбор признаков

Берём только те колонки, которые:
- не являются идентификаторами (`id`, `school_num` как сырой);
- не содержат будущего (никаких `target`, `movement_type`);
- имеют смысл для прогноза на момент принятия решения.

Категориальные признаки передаём в CatBoost нативно.

In [8]:
DROP_COLS = [
    "id", "year", "target",
    "school_num",          # сырой идентификатор
    "movement_type",       # следствие, а не причина
]

feature_cols = [
    c for c in df_model.columns
    if c not in DROP_COLS
]

cat_cols = df_model[feature_cols].select_dtypes(include=["object", "category"]).columns.tolist()
num_cols = [c for c in feature_cols if c not in cat_cols]

print(f"Признаков: {len(feature_cols)} (категориальных: {len(cat_cols)}, числовых: {len(num_cols)})")
print(f"Категориальные: {cat_cols}")

Признаков: 30 (категориальных: 6, числовых: 24)
Категориальные: ['district', 'main_position_cat', 'age_group', 'category_qual_cat', 'speciality', 'academic_degree_cat']


## Временной сплит

Случайный сплит запрещён: данные панельные, один педагог встречается в нескольких годах.  
Правильный подход: обучаемся на годах `<= T`, валидируем на годе `T+1`.

Если лет мало (2–3), делаем один срез. Если больше — можно расширить.

In [9]:
years_sorted = sorted(df_model["year"].unique())
split_year = years_sorted[-1]  # последний доступный год с target → валидация

train = df_model[df_model["year"] < split_year]
valid = df_model[df_model["year"] == split_year]

print(f"Train: {len(train)} строк, годы {train['year'].unique()}")
print(f"Valid: {len(valid)} строк, год {valid['year'].unique()}")
print(f"Доля ухода в train: {train['target'].mean():.3f}")
print(f"Доля ухода в valid: {valid['target'].mean():.3f}")

Train: 17396 строк, годы [2022]
Valid: 17394 строк, год [2023]
Доля ухода в train: 0.149
Доля ухода в valid: 0.132


In [10]:
X_train = train[feature_cols]
y_train = train["target"]

X_valid = valid[feature_cols]
y_valid = valid["target"]

cat_indices = [feature_cols.index(c) for c in cat_cols]

train_pool = Pool(X_train, y_train, cat_features=cat_indices)
valid_pool = Pool(X_valid, y_valid, cat_features=cat_indices)

## Параметры CatBoost

- `iterations=1000` — с запасом, остановим через `early_stopping`.
- `learning_rate=0.05` — умеренный, для стабильности.
- `depth=6` — классика для табличных данных.
- `auto_class_weights='Balanced'` — целевая переменная несбалансирована (уходов мало).
- `eval_metric='AUC'` — главная метрика на валидации.